In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# paths
BASE_DIR = Path().resolve().parent.parent
DATA_RAW = BASE_DIR / "data" / "raw" / "ine"

file_path = DATA_RAW / "pop_residente.xlsx"

print(file_path)

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd().parent   # sobe 1 nível
DATA_RAW = BASE_DIR / "data" / "raw" / "ine"
file_path = DATA_RAW / "pop_residente.xlsx"

print("BASE_DIR:", BASE_DIR)
print("Ficheiro existe?", file_path.exists())


In [ ]:
df = pd.read_excel(file_path)

print("Shape:", df.shape)
df.head()

In [ ]:
df_raw = pd.read_excel(file_path, header=35)

print("Shape:", df_raw.shape)
df_raw.head()

In [ ]:
df = df_raw.copy()

df = df.rename(columns={
    "Código 1": "codigo_ano",
    "Designação 1": "ano",

    "Código 2": "cod_concelho",
    "Designação 2": "concelho",

    "Código 3": "cod_sexo",
    "Designação 3": "sexo",

    "Código 4": "cod_faixa_etaria",
    "Designação 4": "faixa_etaria",

    "Código 5": "cod_escolaridade",
    "Designação 5": "escolaridade",

    "Valor": "valor"
})

df.head()

In [ ]:
cols_keep = [
    "codigo_ano",
    "ano",
    "cod_concelho",
    "concelho",
    "cod_sexo",
    "sexo",
    "cod_faixa_etaria",
    "faixa_etaria",
    "cod_escolaridade",
    "escolaridade",
    "valor"
]

df = df[cols_keep].copy()

df.head()

In [ ]:
df.dtypes

In [ ]:
# Verify the number of rows and duplicates in the expected granularity

print("Linhas totais:", len(df))

chaves_fact = [
    "ano",
    "cod_concelho",
    "cod_sexo",
    "cod_faixa_etaria",
    "cod_escolaridade"
]

duplicados = df.duplicated(subset=chaves_fact).sum()

print("Duplicados na granularidade esperada:", duplicados)

In [ ]:
# Create dim_tempo

dim_tempo = (
    df[["codigo_ano", "ano"]]
    .drop_duplicates()
    .sort_values(["ano", "codigo_ano"])
    .reset_index(drop=True)
)

dim_tempo

In [ ]:
# create dim_concelho

dim_concelho = (
    df[["cod_concelho", "concelho"]]
    .drop_duplicates()
    .sort_values("cod_concelho")
    .reset_index(drop=True)
)

dim_concelho.head(10)

In [ ]:
# create dim_sexo

dim_sexo = (
    df[["cod_sexo", "sexo"]]
    .drop_duplicates()
    .sort_values("cod_sexo")
    .reset_index(drop=True)
)

dim_sexo

In [ ]:
ordem_sexo_map = {
    "1": 1,
    "2": 2,
    "T": 99
}

dim_sexo["ordem_sexo"] = dim_sexo["cod_sexo"].map(ordem_sexo_map)

dim_sexo = dim_sexo.sort_values("ordem_sexo").reset_index(drop=True)

dim_sexo

In [ ]:
# create dim_faixa_etaria

dim_faixa_etaria = (
    df[["cod_faixa_etaria", "faixa_etaria"]]
    .drop_duplicates()
    .sort_values("cod_faixa_etaria")
    .reset_index(drop=True)
)

dim_faixa_etaria.head(20)

In [ ]:
def ordenar_codigo(codigo):
    if codigo == "T":
        return 999
    try:
        return int(codigo)
    except:
        return 998

dim_faixa_etaria["ordem_faixa_etaria"] = dim_faixa_etaria["cod_faixa_etaria"].apply(ordenar_codigo)

dim_faixa_etaria = dim_faixa_etaria.sort_values("ordem_faixa_etaria").reset_index(drop=True)

dim_faixa_etaria

In [ ]:
# create dim_escolaridade

dim_escolaridade = (
    df[["cod_escolaridade", "escolaridade"]]
    .drop_duplicates()
    .sort_values("cod_escolaridade")
    .reset_index(drop=True)
)

dim_escolaridade.head(20)

In [ ]:
dim_escolaridade["ordem_escolaridade"] = dim_escolaridade["cod_escolaridade"].apply(ordenar_codigo)

dim_escolaridade = dim_escolaridade.sort_values("ordem_escolaridade").reset_index(drop=True)

dim_escolaridade

In [ ]:
# Criar fact final without repeated descriptions

fact_pop_residente = df[
    [
        "codigo_ano",
        "ano",
        "cod_concelho",
        "cod_sexo",
        "cod_faixa_etaria",
        "cod_escolaridade",
        "valor"
    ]
].copy()

fact_pop_residente.head()

In [ ]:
# check tables size

print("fact_pop_residente:", fact_pop_residente.shape)
print("dim_tempo:", dim_tempo.shape)
print("dim_concelho:", dim_concelho.shape)
print("dim_sexo:", dim_sexo.shape)
print("dim_faixa_etaria:", dim_faixa_etaria.shape)
print("dim_escolaridade:", dim_escolaridade.shape)

In [ ]:
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

fact_pop_residente.to_parquet(DATA_PROCESSED / "fact_pop_residente.parquet", index=False)
dim_tempo.to_parquet(DATA_PROCESSED / "dim_tempo.parquet", index=False)
dim_concelho.to_parquet(DATA_PROCESSED / "dim_concelho.parquet", index=False)
dim_sexo.to_parquet(DATA_PROCESSED / "dim_sexo.parquet", index=False)
dim_faixa_etaria.to_parquet(DATA_PROCESSED / "dim_faixa_etaria.parquet", index=False)
dim_escolaridade.to_parquet(DATA_PROCESSED / "dim_escolaridade.parquet", index=False)

print("Ficheiros guardados em data/processed/")

In [ ]:
print(dim_concelho.shape)

print("Concelhos duplicados por código:", dim_concelho["cod_concelho"].duplicated().sum())
print("Concelhos duplicados por nome:", dim_concelho["concelho"].duplicated().sum())

dim_concelho.head(20)

In [ ]:
dim_concelho["len_codigo"] = dim_concelho["cod_concelho"].str.len()

dim_concelho["len_codigo"].value_counts().sort_index()

In [ ]:
dim_concelho.sample(10, random_state=48)

In [ ]:
# rename dim_concelho to dim_geografia - general regiosn, not only cities

dim_geografia = dim_concelho.rename(columns={
    "cod_concelho": "cod_geografia",
    "concelho": "geografia"
}).copy()

fact_pop_residente = fact_pop_residente.rename(columns={
    "cod_concelho": "cod_geografia"
}).copy()

dim_geografia.head()

In [ ]:

# normalize codes as text
dim_geografia["cod_geografia"] = dim_geografia["cod_geografia"].astype(str).str.strip()
fact_pop_residente["cod_geografia"] = fact_pop_residente["cod_geografia"].astype(str).str.strip()

dim_geografia.head()

In [ ]:
dim_geografia["len_codigo"] = dim_geografia["cod_geografia"].str.len()

dim_geografia["is_numeric"] = dim_geografia["cod_geografia"].str.match(r"^\d+$", na=False)

dim_geografia["len_codigo"].value_counts().sort_index()

In [ ]:
def classificar_nivel_geografico(codigo):
    codigo = str(codigo).strip()

    if codigo == "PT":
        return "Pais"
    elif len(codigo) == 1:
        return "NUTS1"
    elif len(codigo) == 2:
        return "NUTS2"
    elif len(codigo) == 3:
        return "NUTS3"
    elif len(codigo) == 4:
        return "Concelho"
    else:
        return "Outro"

dim_geografia["nivel_geografico"] = dim_geografia["cod_geografia"].apply(classificar_nivel_geografico)

dim_geografia["nivel_geografico"].value_counts()

In [ ]:
for nivel in sorted(dim_geografia["nivel_geografico"].unique()):
    print(f"\n### {nivel}")
    display(
        dim_geografia.loc[dim_geografia["nivel_geografico"] == nivel, ["cod_geografia", "geografia", "nivel_geografico"]]
        .head(10)
    )

In [ ]:
# Create base columns for future hierarchies (pais, distrito, etc.) - for now with NA as we only have concelhos

dim_geografia["cod_pai"] = pd.NA
dim_geografia["cod_distrito"] = pd.NA

In [ ]:
# create cod_pai based on nivel_geografico

# País
dim_geografia.loc[dim_geografia["nivel_geografico"] == "Pais", "cod_pai"] = pd.NA

# NUTS1 -> pai = PT
dim_geografia.loc[dim_geografia["nivel_geografico"] == "NUTS1", "cod_pai"] = "PT"

# NUTS2 -> pai = 1.º carácter
mask_nuts2 = dim_geografia["nivel_geografico"] == "NUTS2"
dim_geografia.loc[mask_nuts2, "cod_pai"] = dim_geografia.loc[mask_nuts2, "cod_geografia"].str[:1]

# NUTS3 -> pai = 2 primeiros caracteres
mask_nuts3 = dim_geografia["nivel_geografico"] == "NUTS3"
dim_geografia.loc[mask_nuts3, "cod_pai"] = dim_geografia.loc[mask_nuts3, "cod_geografia"].str[:2]

In [ ]:
#create cod_distrito to regions

mask_concelho = dim_geografia["nivel_geografico"] == "Concelho"
dim_geografia.loc[mask_concelho, "cod_distrito"] = dim_geografia.loc[mask_concelho, "cod_geografia"].str[:2]

dim_geografia.head(15)

In [ ]:
map_geografia = dict(zip(dim_geografia["cod_geografia"], dim_geografia["geografia"]))

dim_geografia["geografia_pai"] = dim_geografia["cod_pai"].map(map_geografia)

dim_geografia.head(20)

In [ ]:
print(dim_geografia[["cod_geografia", "geografia", "nivel_geografico", "cod_pai", "geografia_pai", "cod_distrito"]].head(25))

In [ ]:
dim_geografia["cod_pai"].isna().value_counts()

In [ ]:
# get distritos info to enrich dim_geografia

file_distritos = DATA_RAW / "Distritos.xlsx"

df_distritos = pd.read_excel(file_distritos)

print("Shape:", df_distritos.shape)

df_distritos.head(10)

In [ ]:
# normalize and create dim_distrito with info from distritos file

dim_distrito = df_distritos.rename(columns={
    "Código Distrito": "cod_distrito",
    "Distrito": "distrito"
}).copy()

dim_distrito["cod_distrito"] = dim_distrito["cod_distrito"].astype(str).str.zfill(2)
dim_distrito["distrito"] = dim_distrito["distrito"].astype(str).str.strip()

dim_distrito = dim_distrito.drop_duplicates().sort_values("cod_distrito").reset_index(drop=True)

dim_distrito

In [ ]:
# merge dim_distritos with dim_geografia

dim_geografia = dim_geografia.merge(
    dim_distrito,
    on="cod_distrito",
    how="left"
)

dim_geografia.head(20)

In [ ]:
# check by geographic level

dim_geografia.groupby("nivel_geografico")["distrito"].apply(lambda x: x.notna().sum())

In [ ]:
dim_geografia.groupby("nivel_geografico")["distrito"].apply(lambda x: x.notna().sum())

In [ ]:
dim_geografia.to_parquet(DATA_PROCESSED / "dim_geografia.parquet", index=False)
dim_distrito.to_parquet(DATA_PROCESSED / "dim_distrito.parquet", index=False)

print("dim_geografia e dim_distrito guardadas com sucesso.")

In [ ]:
# Final checks

print("Resumo das tabelas geradas:")
print("- fact_pop_residente:", fact_pop_residente.shape)
print("- dim_tempo:", dim_tempo.shape)
print("- dim_geografia:", dim_geografia.shape)
print("- dim_distrito:", dim_distrito.shape)
print("- dim_sexo:", dim_sexo.shape)
print("- dim_faixa_etaria:", dim_faixa_etaria.shape)
print("- dim_escolaridade:", dim_escolaridade.shape)

In [ ]:
# save all tables in Parquet files  

fact_pop_residente.to_parquet(DATA_PROCESSED / "fact_pop_residente.parquet", index=False)
dim_tempo.to_parquet(DATA_PROCESSED / "dim_tempo.parquet", index=False)
dim_geografia.to_parquet(DATA_PROCESSED / "dim_geografia.parquet", index=False)
dim_distrito.to_parquet(DATA_PROCESSED / "dim_distrito.parquet", index=False)
dim_sexo.to_parquet(DATA_PROCESSED / "dim_sexo.parquet", index=False)
dim_faixa_etaria.to_parquet(DATA_PROCESSED / "dim_faixa_etaria.parquet", index=False)
dim_escolaridade.to_parquet(DATA_PROCESSED / "dim_escolaridade.parquet", index=False)

print("Todas as tabelas foram guardadas em data/processed/")